<a href="https://colab.research.google.com/github/pradhapmoorthi/CVND/blob/Consolidated/image_captioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🖼️ Image Captioning — Unified Pipeline (Global/Spatial Attention + Greedy/Beam)

This single notebook **merges** the earlier full pipeline with a **second variant** that adds **spatial attention over CNN feature maps** and **beam search decoding**. Choose behavior with `CFG`:

- `CFG['use_spatial_attention']`: `False` (global feature attention) or `True` (spatial attention over H×W)
- `CFG['decode']`: `'greedy'` or `'beam'`
- `CFG['beam_size']`: beam width when `decode='beam'` (inference only)

**Datasets supported:** Flickr8k (default) and COCO 2017 (optional)

> Tip: Training is compute‑intensive. Prefer a GPU runtime.


## 1) Setup & Configuration

In [1]:
import os, re, io, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple


In [2]:
from google.colab import drive
drive.mount('/content/drive') # Mounts Google Drive to access files

Mounted at /content/drive



## 2) Data Preparation
Ensure the dataset files are arranged as noted in the config. This notebook does **not** auto‑download datasets.


In [3]:
import os, re, io, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision
from PIL import Image
import matplotlib.pyplot as plt

try:
    # Import BLEU score for evaluation if NLTK is available
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    nltk_ok = True
except Exception:
    nltk_ok = False

SEED = 42
random.seed(SEED) # Set random seed for Python's random module
np.random.seed(SEED) # Set random seed for NumPy
torch.manual_seed(SEED) # Set random seed for PyTorch CPU operations
torch.cuda.manual_seed_all(SEED) # Set random seed for PyTorch CUDA (GPU) operations

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Determine computing device (GPU if available, else CPU)
print('Device:', device)

SPECIAL_TOKENS = {'pad':'<pad>', 'bos':'<bos>', 'eos':'<eos>', 'unk':'<unk>'} # Define special tokens for vocabulary

CFG = {
    'dataset': 'COCO',          # 'Flickr8k' or 'COCO'
    'data_root': '/data',

    # Flickr8k expected structure:
    #   ./data/Flickr8k/images/
    #   ./data/Flickr8k/captions/Flickr8k.token.txt
    #   ./data/Flickr8k/captions/Flickr_8k.trainImages.txt
    #   ./data/Flickr8k/captions/Flickr_8k.devImages.txt
    #   ./data/Flickr8k/captions/Flickr_8k.testImages.txt

    # COCO expected structure:
    #   ./data/COCO/train2017/*.jpg
    #   ./data/COCO/val2017/*.jpg
    #   ./data/COCO/annotations/captions_train2017.json
    #   ./data/COCO/annotations/captions_val2017.json

    'min_freq': 5, # Minimum frequency for a word to be included in the vocabulary
    'max_len': 20, # Maximum caption length
    'batch_size': 64,
    'num_workers': 2,

    # Model & training configurations
    'use_spatial_attention': False,   # Toggle between global and spatial attention
    'encoder_cnn': 'resnet50',
    'embed_dim': 256,
    'hidden_dim': 512,
    'attention_dim': 256,
    'dropout': 0.3,

    'epochs': 4,            # Number of training epochs (increase for real training)
    'lr': 3e-4, # Learning rate
    'clip': 1.0, # Gradient clipping value
    'teacher_forcing': 0.5, # Teacher forcing ratio for decoder training

    # Decoding strategy
    'decode': 'greedy',       # 'greedy' or 'beam' search decoding
    'beam_size': 3,         # Beam width when decode='beam' (inference only)

    'save_dir': '/content/drive/MyDrive/image_captioning_checkpoints', # Directory to save model checkpoints
    'exp_name': 'captioning_nonspatial_greedy', # Experiment name for saving files
    'fp16': True, # Enable mixed precision training
}

os.makedirs(CFG['save_dir'], exist_ok=True) # Create the save directory if it doesn't exist


Device: cuda


In [4]:
import os, re, io, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple

SOURCE_TRAIN_ZIP_PATH = '/content/drive/MyDrive/datasets/COCO/train2017.zip' # <--- UPDATE THESE PATHS
SOURCE_VAL_ZIP_PATH = '/content/drive/MyDrive/datasets/COCO/val2017.zip'
SOURCE_ANN_ZIP_PATH = '/content/drive/MyDrive/datasets/COCO/annotations_trainval2017.zip'

DEST_DIR = Path('/data/COCO') # Define the destination directory for extracted COCO dataset

# Ensure the base destination directory exists
DEST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ensured base directory '{DEST_DIR}' exists.")

def copy_and_extract_zip(source_path, dest_dir, extract_to_subdir=None):
    # Check if source zip file exists
    if not Path(source_path).exists():
        print(f"Source file '{source_path}' does not exist. Skipping.")
        return

    zip_filename = Path(source_path).name
    dest_zip_path = dest_dir / zip_filename

    # Copy the zip file from source to destination if not already present
    if not dest_zip_path.exists():
        print(f"Copying {source_path} to {dest_zip_path}...")
        !cp "{source_path}" "{dest_zip_path}"
        print("Copy complete.")
    else:
        print(f"'{dest_zip_path}' already exists. Skipping copy.")

    # Extract the zip file
    if dest_zip_path.exists():
        print(f"'{dest_zip_path}' successfully copied.")

        # Determine the target directory for extraction
        extract_target_dir = dest_dir / (extract_to_subdir if extract_to_subdir else zip_filename.split('.')[0])

        # Always force re-extraction if the directory exists for COCO, as previous heuristics might fail.
        # This is a more aggressive approach to ensure a clean slate after issues.
        force_re_extract = False
        if extract_target_dir.is_dir():
            print(f"Directory '{extract_target_dir}' found. Forcing re-extraction for reliability...")
            force_re_extract = True

        if force_re_extract:
            print(f"Removing '{extract_target_dir}' for re-extraction.")
            !rm -rf "{extract_target_dir}" # Use -rf to ensure removal of non-empty directory

        if not extract_target_dir.exists():
            extract_target_dir.mkdir(parents=True, exist_ok=True) # Create extraction directory
            print(f"Created extraction directory: {extract_target_dir}")

            print(f"Extracting {dest_zip_path} to {extract_target_dir}...")
            !unzip -q "{dest_zip_path}" -d "{extract_target_dir}" # Unzip the file silently
            print("Extraction complete.")

            # Custom handling for COCO annotations if they are nested
            if extract_to_subdir == 'annotations':
                nested_annotations_path = extract_target_dir / 'annotations'
                if nested_annotations_path.is_dir() and any(nested_annotations_path.iterdir()):
                    print(f"Found nested annotations directory: {nested_annotations_path}. Moving contents...")
                    # Use rsync to move contents more robustly
                    !rsync -a "{nested_annotations_path}/" "{extract_target_dir}"/
                    !rm -rf "{nested_annotations_path}" # Use rm -rf here
                    print("Moved nested annotations contents.")
            # Custom handling for COCO images if they are nested (e.g., train2017.zip extracts to train2017/train2017)
            elif extract_to_subdir in ['train2017', 'val2017']:
                nested_image_path = extract_target_dir / extract_to_subdir
                if nested_image_path.is_dir() and any(nested_image_path.iterdir()):
                    print(f"Found nested image directory: {nested_image_path}. Moving contents...")
                    # Use rsync to move contents more robustly
                    !rsync -a "{nested_image_path}/" "{extract_target_dir}"/
                    !rm -rf "{nested_image_path}" # Use rm -rf here
                    print("Moved nested image contents.")

            # Optional: Remove the copied zip file to save space
            print(f"Removing {dest_zip_path}...")
            !rm "{dest_zip_path}" # Remove the zip file after extraction
            print("Zip file removed.")
        else:
            # This 'else' block should ideally not be reached if force_re_extract is True
            print(f"Extraction directory '{extract_target_dir}' already exists, but was supposed to be removed. Something went wrong.")
    else:
        print(f"Error: '{dest_zip_path}' not found after copy. Please ensure {source_path} is correct and try again.")

# Call the function to copy and extract training images
copy_and_extract_zip(SOURCE_TRAIN_ZIP_PATH, DEST_DIR, 'train2017')
# Call the function to copy and extract validation images
copy_and_extract_zip(SOURCE_VAL_ZIP_PATH, DEST_DIR, 'val2017')
# Call the function to copy and extract annotations
copy_and_extract_zip(SOURCE_ANN_ZIP_PATH, DEST_DIR, 'annotations')

# Verify the dataset structure after extraction
print("\nVerifying COCO dataset structure:")
print(f"Is {DEST_DIR / 'train2017'} directory present? { (DEST_DIR / 'train2017').is_dir() }")
print(f"Is {DEST_DIR / 'val2017'} directory present? { (DEST_DIR / 'val2017').is_dir() }")
print(f"Is {DEST_DIR / 'annotations'} directory present? { (DEST_DIR / 'annotations').is_dir() }")
print(f"Is {DEST_DIR / 'annotations' / 'captions_train2017.json'} present? { (DEST_DIR / 'annotations' / 'captions_train2017.json').is_file() }")
print(f"Is {DEST_DIR / 'annotations' / 'captions_val2017.json'} present? { (DEST_DIR / 'annotations' / 'captions_val2017.json').is_file() }")


Ensured base directory '/data/COCO' exists.
Copying /content/drive/MyDrive/datasets/COCO/train2017.zip to /data/COCO/train2017.zip...
Copy complete.
'/data/COCO/train2017.zip' successfully copied.
Created extraction directory: /data/COCO/train2017
Extracting /data/COCO/train2017.zip to /data/COCO/train2017...
Extraction complete.
Found nested image directory: /data/COCO/train2017/train2017. Moving contents...
Moved nested image contents.
Removing /data/COCO/train2017.zip...
Zip file removed.
Copying /content/drive/MyDrive/datasets/COCO/val2017.zip to /data/COCO/val2017.zip...
Copy complete.
'/data/COCO/val2017.zip' successfully copied.
Created extraction directory: /data/COCO/val2017
Extracting /data/COCO/val2017.zip to /data/COCO/val2017...
Extraction complete.
Found nested image directory: /data/COCO/val2017/val2017. Moving contents...
Moved nested image contents.
Removing /data/COCO/val2017.zip...
Zip file removed.
Copying /content/drive/MyDrive/datasets/COCO/annotations_trainval201

## 3) Vocabulary & Tokenization

In [5]:
class Vocabulary:
    """
    A class to manage the vocabulary, including tokenization, building the vocabulary
    from a corpus, and converting between tokens and numerical IDs.
    """
    def __init__(self, min_freq=5):
        """
        Initializes the Vocabulary object.
        Args:
            min_freq (int): The minimum frequency a word must have to be included in the vocabulary.
        """
        self.min_freq = min_freq # Minimum frequency for a word to be included in the vocabulary
        self.freqs = {} # Dictionary to store word frequencies
        self.stoi = {} # Dictionary mapping string tokens to integer IDs
        self.itos = [] # List mapping integer IDs back to string tokens

    @staticmethod
    def tokenize(text: str) -> List[str]:
        """
        Tokenizes a given string of text into a list of words, numbers, and symbols.
        Args:
            text (str): The input string to be tokenized.
        Returns:
            List[str]: A list of tokens (strings).
        """
        # Convert text to lowercase and tokenize using regex to capture words, numbers, and symbols
        text = text.lower()
        return re.findall(r"[a-zA-Z]+|\d+|[^\s\w]", text)

    def build(self, sentences: List[str]):
        """
        Builds the vocabulary from a list of sentences. It calculates word frequencies and
        populates the stoi and itos mappings based on the min_freq.
        Args:
            sentences (List[str]): A list of strings, where each string is a caption or sentence.
        """
        # Calculate word frequencies from the given sentences
        for s in sentences:
            for tok in self.tokenize(s):
                self.freqs[tok] = self.freqs.get(tok, 0) + 1
        # Initialize itos with special tokens
        self.itos = [SPECIAL_TOKENS['pad'], SPECIAL_TOKENS['bos'], SPECIAL_TOKENS['eos'], SPECIAL_TOKENS['unk']]
        # Initialize stoi with special tokens and their IDs
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        # Add words to vocabulary if their frequency meets min_freq
        for tok, f in sorted(self.freqs.items(), key=lambda x: (-x[1], x[0])):
            if f >= self.min_freq and tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)
        print(f"Built vocab size: {len(self)}")

    def __len__(self):
        """
        Returns the total number of unique tokens in the vocabulary.
        Returns:
            int: The size of the vocabulary.
        """
        return len(self.itos) # Returns the size of the vocabulary

    def numericalize(self, tokens: List[str]) -> List[int]:
        """
        Converts a list of string tokens into a list of their corresponding numerical IDs.
        Args:
            tokens (List[str]): A list of string tokens.
        Returns:
            List[int]: A list of numerical IDs. Unknown tokens are mapped to the <unk> token's ID.
        """
        # Convert a list of tokens into a list of numerical IDs
        return [self.stoi.get(t, self.stoi[SPECIAL_TOKENS['unk']]) for t in tokens]

    def denumericalize(self, ids: List[int]) -> List[str]:
        """
        Converts a list of numerical IDs back into a list of their corresponding string tokens.
        Args:
            ids (List[int]): A list of numerical token IDs.
        Returns:
            List[str]: A list of string tokens.
        """
        # Convert a list of numerical IDs into a list of string tokens
        return [self.itos[i] for i in ids]


def save_vocab(vocab: Vocabulary, path: str):
    """
    Saves the Vocabulary object to a JSON file, including its min_freq and itos list.
    Args:
        vocab (Vocabulary): The Vocabulary object to be saved.
        path (str): The file path where the vocabulary will be saved.
    """
    # Save the vocabulary (min_freq and itos list) to a JSON file
    obj = {'min_freq': vocab.min_freq, 'itos': vocab.itos}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f)


def load_vocab(path: str) -> Vocabulary:
    """
    Loads a Vocabulary object from a specified JSON file.
    Args:
        path (str): The file path from which to load the vocabulary.
    Returns:
        Vocabulary: The loaded Vocabulary object.
    """
    # Load the vocabulary from a JSON file
    with open(path, 'r', encoding='utf-8') as f:
        obj = json.load(f)
    v = Vocabulary(min_freq=obj['min_freq'])
    v.itos = obj['itos']
    v.stoi = {tok: i for i, tok in enumerate(v.itos)}
    return v


## 4) Dataset, Transforms, & DataLoader

In [6]:
class Flickr8kCaptions:
    """
    A class to manage the Flickr8k dataset, handling paths and caption loading.
    """
    def __init__(self, root: str):
        """
        Initializes the Flickr8kCaptions class.
        Args:
            root (str): The root directory where the Flickr8k dataset is located.
        """
        root = Path(root)
        self.images_dir = root / 'images'
        self.captions_dir = root / 'captions'
        self.token_file = self.captions_dir / 'Flickr8k.token.txt'
        self.train_file = self.captions_dir / 'Flickr_8k.trainImages.txt'
        self.dev_file   = self.captions_dir / 'Flickr_8k.devImages.txt'
        self.test_file  = self.captions_dir / 'Flickr_8k.testImages.txt'
        assert self.token_file.exists(), f"Missing {self.token_file}"
        self.image2caps = {} # Dictionary to store image to captions mapping
        with open(self.token_file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split(' ')
                if len(parts) != 2: continue
                img_id, cap = parts
                img = img_id.split('#')[0]
                self.image2caps.setdefault(img, []).append(cap)
        def read_list(p):
            with open(p, 'r', encoding='utf-8') as f:
                return [x.strip() for x in f if x.strip()]
        # Read image file lists for train, validation, and test splits
        self.train_imgs = read_list(self.train_file)
        self.val_imgs   = read_list(self.dev_file)
        self.test_imgs  = read_list(self.test_file)

class COCOV2:
    """
    A class to manage the COCO 2017 dataset, handling paths and caption loading.
    """
    def __init__(self, root: str):
        """
        Initializes the COCOV2 class.
        Args:
            root (str): The root directory where the COCO dataset is located.
        """
        root = Path(root)
        self.train_dir = root / 'train2017' # Path to training images
        self.val_dir    = root / 'val2017'   # Path to validation images
        ann_dir = root / 'annotations'     # Path to annotations directory
        # Load COCO training captions JSON file
        with open(ann_dir / 'captions_train2017.json', 'r') as f:
            train_ann = json.load(f)
        # Load COCO validation captions JSON file
        with open(ann_dir / 'captions_val2017.json', 'r') as f:
            val_ann = json.load(f)
        def build(ann, img_dir):
            """
            Helper function to build mappings from image IDs to filenames and filenames to captions.
            Args:
                ann (dict): The annotation dictionary (e.g., from captions_train2017.json).
                img_dir (Path): The directory containing the images.
            Returns:
                Tuple[Path, dict, list]: Image directory, filename-to-captions map, and list of files.
            """
            # Build mapping from image ID to filename
            id2file = {img['id']: img['file_name'] for img in ann['images']}
            file2caps = {} # Dictionary to store filename to captions mapping
            # Populate file2caps with captions for each image
            for a in ann['annotations']:
                fn = id2file[a['image_id']]
                file2caps.setdefault(fn, []).append(a['caption'])
            # Filter files to only include those that actually exist in the image directory
            files = [f for f in file2caps.keys() if (img_dir / f).exists()]
            return img_dir, file2caps, files
        # Build data structures for training and validation sets
        self.train_dir, self.train_caps, self.train_files = build(train_ann, self.train_dir)
        self.val_dir,   self.val_caps,   self.val_files   = build(val_ann, self.val_dir)

class ImageCaptionDataset(Dataset):
    """
    A PyTorch Dataset for loading image-caption pairs.
    """
    def __init__(self, dataset_name: str, split: str, cfg: Dict, vocab: Vocabulary=None):
        """
        Initializes the ImageCaptionDataset.
        Args:
            dataset_name (str): Name of the dataset ('Flickr8k' or 'COCO').
            split (str): The dataset split ('train', 'val', or 'test').
            cfg (Dict): Configuration dictionary containing dataset parameters like max_len, data_root.
            vocab (Vocabulary, optional): The vocabulary object used for tokenization.
        """
        self.dataset_name = dataset_name
        self.split = split
        self.cfg = cfg
        self.vocab = vocab
        self.max_len = cfg['max_len'] # Maximum caption length

        pairs = [] # List to store (image_path, caption) tuples
        # Load image-caption pairs based on the specified dataset and split
        if dataset_name.lower() == 'flickr8k':
            flickr = Flickr8kCaptions(Path(cfg['data_root']) / 'Flickr8k')
            files = flickr.train_imgs if split=='train' else (flickr.val_imgs if split=='val' else flickr.test_imgs)
            for fn in files:
                for cap in flickr.image2caps.get(fn, []):
                    pairs.append((flickr.images_dir / fn, cap))
        elif dataset_name.lower() == 'coco':
            coco = COCOV2(Path(cfg['data_root']) / 'COCO')
            if split=='train':
                files, caps_map, base = coco.train_files, coco.train_caps, coco.train_dir
            else:
                files, caps_map, base = coco.val_files, coco.val_caps, coco.val_dir
            for fn in files:
                for cap in caps_map.get(fn, []):
                    pairs.append((base / fn, cap))
        else:
            raise ValueError('Unknown dataset: ' + dataset_name)
        self.pairs = pairs
        print(f"Loaded {len(self.pairs)} pairs for {dataset_name} [{split}]")

        # Define image transformations for training and validation/test splits
        if split=='train':
            self.tfms = transforms.Compose([
                transforms.Resize((256,256)),
                transforms.RandomCrop((224,224)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
            ])
        else:
            self.tfms = transforms.Compose([
                transforms.Resize((224,224)),
                transforms.CenterCrop((224,224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
            ])

    def __len__(self):
        """
        Returns the total number of image-caption pairs in the dataset.
        Returns:
            int: The number of pairs.
        """
        return len(self.pairs) # Returns the total number of image-caption pairs

    def __getitem__(self, idx):
        """
        Retrieves an image, its numericalized caption, and the caption's length for a given index.
        Args:
            idx (int): The index of the item to retrieve.
        Returns:
            Tuple[torch.Tensor, torch.Tensor, int]:
                img (torch.Tensor): The transformed image tensor.
                ids (torch.Tensor): A tensor of numericalized tokens for the caption, including BOS and EOS tokens.
                length (int): The actual length of the tokenized caption.
        """
        path, caption = self.pairs[idx]
        img = Image.open(path).convert('RGB') # Load image and convert to RGB
        img = self.tfms(img) # Apply transformations to the image
        # Tokenize caption and add special BOS and EOS tokens
        tokens = [SPECIAL_TOKENS['bos']] + self.vocab.tokenize(caption) + [SPECIAL_TOKENS['eos']]
        ids = self.vocab.numericalize(tokens) # Convert tokens to numerical IDs
        # Truncate caption if it exceeds max_len and ensure EOS token is present
        if len(ids) > self.max_len:
            ids = ids[:self.max_len]
            if ids[-1] != self.vocab.stoi[SPECIAL_TOKENS['eos']]:
                ids[-1] = self.vocab.stoi[SPECIAL_TOKENS['eos']]
        length = len(ids) # Actual length of the caption
        return img, torch.tensor(ids, dtype=torch.long), length # Return image, caption IDs, and length

def pad_collate(batch):
    """
    A custom collate function for DataLoader to handle variable-length caption sequences
    by padding them to the maximum length within each batch.
    Args:
        batch (List[Tuple]): A list of tuples, where each tuple is (img, seq, length) from ImageCaptionDataset.__getitem__.
    Returns:
        Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
            imgs (torch.Tensor): A stacked tensor of images for the batch.
            padded (torch.Tensor): A tensor of padded caption sequences, all of the same length, filled with pad_id (0).
            lens (torch.Tensor): A tensor of the original lengths of the captions before padding.
    """
    # Custom collate function to pad sequences in a batch to the maximum length
    imgs, seqs, lens = zip(*batch)
    imgs = torch.stack(imgs, dim=0) # Stack images into a single tensor
    max_len = max(lens) # Find the maximum sequence length in the current batch
    pad_id = 0 # Padding token ID (assuming <pad> is 0)
    # Create a padded tensor filled with pad_id
    padded = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    # Copy original sequences into the padded tensor
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = s
    lens = torch.tensor(lens, dtype=torch.long) # Convert lengths to a tensor
    return imgs, padded, lens # Return batched images, padded sequences, and their lengths


### Build/Load Vocabulary

In [7]:

vocab_path = Path(CFG['save_dir']) / f"{CFG['dataset'].lower()}_vocab.json" # Define path for vocabulary file
# Check if vocabulary file already exists
if vocab_path.exists():
    vocab = load_vocab(str(vocab_path)) # Load existing vocabulary
    print(f"Loaded vocab from {vocab_path} (size={len(vocab)})")
else:
    corpus = [] # List to collect all training captions
    # Load captions based on the dataset configuration
    if CFG['dataset'].lower() == 'flickr8k':
        flickr = Flickr8kCaptions(Path(CFG['data_root']) / 'Flickr8k')
        for fn in flickr.train_imgs:
            corpus.extend(flickr.image2caps.get(fn, []))
    else:
        coco = COCOV2(Path(CFG['data_root']) / 'COCO')
        for fn in coco.train_files:
            corpus.extend(coco.train_caps.get(fn, []))
    vocab = Vocabulary(min_freq=CFG['min_freq']) # Initialize vocabulary with min_freq from config
    vocab.build(corpus) # Build vocabulary from the collected corpus
    save_vocab(vocab, str(vocab_path)) # Save the newly built vocabulary
    print(f"Saved vocab to {vocab_path}")


Loaded vocab from /content/drive/MyDrive/image_captioning_checkpoints/coco_vocab.json (size=10217)


### DataLoaders

In [8]:
train_ds = ImageCaptionDataset(CFG['dataset'], 'train', CFG, vocab) # Initialize training dataset
val_ds   = ImageCaptionDataset(CFG['dataset'], 'val',   CFG, vocab) # Initialize validation dataset

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], collate_fn=pad_collate, pin_memory=True) # Create DataLoader for training
val_loader   = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], collate_fn=pad_collate, pin_memory=True) # Create DataLoader for validation

len(train_loader), len(val_loader) # Print the number of batches in train and validation loaders


Loaded 591753 pairs for COCO [train]
Loaded 25014 pairs for COCO [val]


(9247, 391)

## 5) Models: Encoder/Decoder + Attention (Global & Spatial)

In [9]:
import os, re, io, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision
from PIL import Image
import matplotlib.pyplot as plt

try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    nltk_ok = True
except Exception:
    nltk_ok = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

SPECIAL_TOKENS = {'pad':'<pad>', 'bos':'<bos>', 'eos':'<eos>', 'unk':'<unk>'}

# -------- Spatial feature encoder & attention --------
class EncoderCNN_Spatial(nn.Module):
    """
    CNN-based encoder that extracts spatial feature maps from images for spatial attention.
    """
    def __init__(self):
        """
        Initializes the EncoderCNN_Spatial.
        """
        super().__init__()
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2) # Load pre-trained ResNet-50
        self.cnn = nn.Sequential(*list(m.children())[:-2])  # Bx2048x7x7 # Remove last two layers to get convolutional feature maps
        self.adapt = nn.Conv2d(2048, 512, kernel_size=1) # 1x1 convolution to reduce feature map depth

    def forward(self, images):
        """
        Processes input images through the spatial CNN encoder to extract spatial feature maps.
        Args:
            images (torch.Tensor): A batch of input images.
        Returns:
            Tuple[torch.Tensor, Tuple[int, int]]:
                seq (torch.Tensor): A sequence of spatial features, reshaped to (batch_size, H*W, channels).
                (H,W) (Tuple[int, int]): The height and width of the feature maps.
        """
        fmap = self.cnn(images)     # B,2048,7,7 # Get feature maps from CNN
        fmap = self.adapt(fmap)     # B,512,7,7 # Apply 1x1 convolution
        B,C,H,W = fmap.shape # Get batch size, channels, height, width
        seq = fmap.permute(0,2,3,1).contiguous().view(B, H*W, C)  # B,T(=49),512 # Reshape feature map for attention (B, H*W, C)
        return seq, (H,W) # Return sequence of features and original H,W dimensions

class SpatialAttention(nn.Module):
    """
    Spatial attention mechanism that focuses on different regions of an image.
    """
    def __init__(self, feat_dim, hidden_dim):
        """
        Initializes the SpatialAttention.
        Args:
            feat_dim (int): The dimension of the spatial feature vectors.
            hidden_dim (int): The dimension of the decoder's hidden state.
        """
        super().__init__()
        self.W = nn.Linear(feat_dim, hidden_dim) # Linear layer for input features
        self.U = nn.Linear(hidden_dim, hidden_dim) # Linear layer for hidden state
        self.v = nn.Linear(hidden_dim, 1) # Linear layer to compute attention scores

    def forward(self, feats, hidden):
        """
        Computes spatial attention weights and a context vector from spatial features and decoder hidden state.
        Args:
            feats (torch.Tensor): Spatial features (B, T, C).
            hidden (torch.Tensor): The decoder's current hidden state (B, H).
        Returns:
            Tuple[torch.Tensor, torch.Tensor]:
                ctx (torch.Tensor): The context vector (B, C).
                alpha (torch.Tensor): The attention weights (B, T, 1).
        """
        # feats: B,T,C; hidden: B,H
        # Compute attention scores
        score = self.v(torch.tanh(self.W(feats) + self.U(hidden).unsqueeze(1)))  # B,T,1
        alpha = torch.softmax(score, dim=1)                                       # B,T,1 # Apply softmax for attention weights
        ctx = (alpha * feats).sum(1)                                              # B,C # Compute context vector
        return ctx, alpha

class Decoder_Spatial(nn.Module):
    """
    Decoder for spatial attention, generating captions word by word.
    """
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, feat_dim=512, dropout=0.3):
        """
        Initializes the Decoder_Spatial.
        Args:
            vocab_size (int): The size of the vocabulary.
            embed_dim (int): The dimension of word embeddings.
            hidden_dim (int): The dimension of the LSTM hidden state.
            feat_dim (int): The dimension of the spatial feature vectors.
            dropout (float): Dropout probability for regularization.
        """
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0) # Embedding layer for tokens
        self.attn  = SpatialAttention(feat_dim, hidden_dim) # Spatial attention mechanism
        self.lstm  = nn.LSTMCell(embed_dim + feat_dim, hidden_dim) # LSTM cell
        self.fc    = nn.Linear(hidden_dim, vocab_size) # Fully connected layer for token prediction
        self.drop  = nn.Dropout(dropout) # Dropout layer
        self.hidden_dim = hidden_dim

    def forward(self, feats, captions):
        """
        Performs a forward pass during training for the spatial decoder.
        Args:
            feats (torch.Tensor): Spatial features from the encoder (B, H*W, C).
            captions (torch.Tensor): Ground truth captions (B, T).
        Returns:
            torch.Tensor: Logits for each token prediction at each time step (B, T-1, vocab_size).
        """
        B,T = captions.size()
        # Initialize hidden and cell states
        h = feats.new_zeros((B, self.hidden_dim))
        c = feats.new_zeros((B, self.hidden_dim))
        inp = self.embed(captions[:,0]) # Embedding for the first token
        outs=[]
        for t in range(1,T):
            ctx,_ = self.attn(feats, h) # Compute context vector with spatial attention
            h,c = self.lstm(torch.cat([inp, ctx], dim=1), (h,c)) # Update LSTM states
            logits = self.fc(self.drop(h)) # Predict logits
            outs.append(logits.unsqueeze(1))
            inp = self.embed(captions[:,t]) # Embedding for the next token
        return torch.cat(outs, dim=1)

    def greedy_decode(self, feats, bos_id, eos_id, max_len=20):
        """
        Generates captions using a greedy search strategy with spatial attention.
        Args:
            feats (torch.Tensor): Spatial features from the encoder (B, H*W, C).
            bos_id (int): ID of the Begin-Of-Sentence token.
            eos_id (int): ID of the End-Of-Sentence token.
            max_len (int): Maximum length of the generated caption.
        Returns:
            Tuple[List[List[int]], List[torch.Tensor]]:
                List[List[int]]: A list of decoded token ID sequences.
                List[torch.Tensor]: A list of attention weights (B, H*W) for each generated token.
        """
        B = feats.size(0)
        # Initialize hidden and cell states
        h = feats.new_zeros((B, self.hidden_dim))
        c = feats.new_zeros((B, self.hidden_dim))
        # Start with BOS token
        x = torch.full((B,), bos_id, dtype=torch.long, device=feats.device)
        emb = self.embed(x)
        seqs=[]; alphas=[] # Lists to store sequences and attention weights
        for _ in range(max_len):
            ctx,alpha = self.attn(feats, h) # Compute context vector and attention weights
            h,c = self.lstm(torch.cat([emb, ctx], dim=1), (h,c)) # Update LSTM states
            logit = self.fc(h)
            x = logit.argmax(-1) # Greedy token prediction
            seqs.append(x)
            alphas.append(alpha.squeeze(-1))  # B,T # Store attention weights
            emb = self.embed(x)
        # stop at eos per sample
        out=[]
        for b in range(B):
            toks=[]
            for t in seqs:
                tok=t[b].item()
                if tok==eos_id: break
                toks.append(tok)
            out.append(toks)
        return out, alphas  # list of tokens per sample, and list of alpha tensors per step

    def beam_search(self, feats, bos_id, eos_id, beam=3, max_len=20):
        """
        Generates a single caption using beam search, which explores multiple high-probability sequences.
        Args:
            feats (torch.Tensor): Spatial features from the encoder (batch_size=1, H*W, C).
            bos_id (int): ID of the Begin-Of-Sentence token.
            eos_id (int): ID of the End-Of-Sentence token.
            beam (int): The beam width (number of sequences to keep at each step).
            max_len (int): Maximum length of the generated caption.
        Returns:
            List[int]: The best decoded token ID sequence.
        Note: This implementation currently supports a batch size of 1.
        """
        # NOTE: supports B==1 for simplicity
        assert feats.size(0) == 1, 'Beam search currently supports batch size 1.'
        # Initialize hidden and cell states
        h = feats.new_zeros((1, self.hidden_dim))
        c = feats.new_zeros((1, self.hidden_dim))
        # Initialize sequences for beam search: (tokens, logprob, h, c)
        sequences = [([bos_id], 0.0, h, c)]
        for _ in range(max_len):
            new_list = []
            for toks,score,hx,cx in sequences:
                if toks[-1] == eos_id:
                    new_list.append((toks, score, hx, cx))
                    continue
                x = torch.tensor([toks[-1]], device=feats.device) # Current token
                emb = self.embed(x) # Embedding for the current token
                ctx,_ = self.attn(feats, hx) # Compute context vector
                hx, cx = self.lstm(torch.cat([emb, ctx], dim=1), (hx, cx)) # Update LSTM states
                logits = self.fc(hx) # Predict logits
                logprobs = F.log_softmax(logits, dim=-1) # Convert logits to log probabilities
                topk = torch.topk(logprobs, beam) # Get top 'beam' probable next tokens
                for i in range(beam):
                    tok = int(topk.indices[0, i].item())
                    sc  = float(score + topk.values[0, i].item())
                    new_list.append((toks + [tok], sc, hx.clone(), cx.clone())) # Add new sequences to the list
            # Prune sequences to keep only the top 'beam' sequences based on log probability
            new_list.sort(key=lambda x: x[1], reverse=True)
            sequences = new_list[:beam]
        best = sequences[0][0] # Get the best sequence from beam search
        # strip BOS and cut at EOS
        out=[]
        for t in best[1:]:
            if t==eos_id: break # Stop at EOS token
            out.append(t)
        return out

Device: cuda


### Build Models per Config

In [10]:
# Model factory
# Initialize encoder and decoder based on the configuration for spatial attention
encoder = EncoderCNN_Spatial().to(device) # Spatial encoder
decoder = Decoder_Spatial(
    vocab_size=len(vocab),
    embed_dim=CFG['embed_dim'],
    hidden_dim=CFG['hidden_dim'],
    feat_dim=512,
    dropout=CFG['dropout']).to(device) # Spatial decoder

# Combine parameters from decoder and trainable encoder parts for optimization
params = list(decoder.parameters()) + [p for p in encoder.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=CFG['lr']) # Adam optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0) # Cross-entropy loss, ignoring padding token
scaler = torch.amp.GradScaler('cuda', enabled=CFG['fp16']) # Gradient scaler for mixed precision training

ckpt_path = Path(CFG['save_dir']) / f"{CFG['exp_name']}.pt" # Define checkpoint save path

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 221MB/s]


## 6) Training & Evaluation (BLEU‑4)

In [3]:
import os # Add this line to import the os module

def train_one_epoch(epoch):
    """
    Conducts a single training epoch for the image captioning model.
    Args:
        epoch (int): The current epoch number.
    Returns:
        float: The average training loss for the epoch.
    """
    encoder.train(); decoder.train() # Set models to training mode
    total = 0.0 # Initialize total loss for the epoch
    step_group_start_time = time.time() # Initialize timer for step groups
    for i,(imgs,caps,lens) in enumerate(train_loader):
        imgs, caps = imgs.to(device), caps.to(device) # Move data to appropriate device
        optimizer.zero_grad() # Clear gradients
        with torch.cuda.amp.autocast(enabled=CFG['fp16']):
            feats, _ = encoder(imgs) # Encode images with spatial encoder
            logits = decoder(feats, caps) # Decode captions with spatial decoder
            targets = caps[:,1:logits.size(1)+1] # Prepare target captions (shifted by one for prediction)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1)) # Calculate loss
        scaler.scale(loss).backward() # Scale loss and perform backward pass
        nn.utils.clip_grad_norm_(params, CFG['clip']) # Clip gradients to prevent exploding gradients
        scaler.step(optimizer) # Update optimizer weights
        scaler.update() # Update the scaler for the next iteration
        total += loss.item() # Accumulate total loss
        if (i+1)%50==0:
            elapsed_time = time.time() - step_group_start_time # Calculate elapsed time for 50 steps
            print(f"epoch {epoch} step {i+1}/{len(train_loader)} loss {total/(i+1):.4f} (last 50 steps took {elapsed_time:.2f}s)")
            step_group_start_time = time.time() # Reset timer for next 50 steps
    return total/max(1,len(train_loader)) # Return average loss for the epoch


def evaluate_bleu(sample_limit=1000):
    """
    Evaluates the model's performance on the validation set using the BLEU-4 metric.
    Args:
        sample_limit (int): The maximum number of samples from the validation set to evaluate.
    Returns:
        float: The calculated BLEU-4 score.
    """
    encoder.eval(); decoder.eval() # Set models to evaluation mode
    refs, hyps = [], [] # Lists to store reference and hypothesis captions
    with torch.no_grad(): # Disable gradient calculations during evaluation
        count=0
        for imgs, caps, lens in val_loader:
            imgs = imgs.to(device)
            feats,_ = encoder(imgs) # Encode images
            # Decode captions using greedy search for spatial attention
            # Note: Evaluation is typically done with greedy search for consistent comparison, even if beam search is used for inference
            out_ids,_ = decoder.greedy_decode(feats, vocab.stoi[SPECIAL_TOKENS['bos']], vocab.stoi[SPECIAL_TOKENS['eos']], max_len=CFG['max_len'])
            for b in range(len(out_ids)):
                tgt_ids = caps[b].tolist() # Convert target caption IDs to list
                # strip bos/eos/pad tokens from reference caption
                try: bos = tgt_ids.index(vocab.stoi[SPECIAL_TOKENS['bos']])
                except ValueError: bos=0
                eos = tgt_ids.index(vocab.stoi[SPECIAL_TOKENS['eos']]) if vocab.stoi[SPECIAL_TOKENS['eos']] in tgt_ids else len(tgt_ids)
                ref = vocab.denumericalize(tgt_ids[bos+1:eos]) # Denumericalize reference caption
                hyp = vocab.denumericalize(out_ids[b]) # Denumericalize hypothesized caption
                if ref and hyp:
                    refs.append([ref]) # Add reference caption
                    hyps.append(hyp) # Add hypothesized caption
            count += len(out_ids)
            if count>=sample_limit: break # Break if sample limit is reached
    if nltk_ok and hyps:
        smoothie = SmoothingFunction().method4 # Smoothing function for BLEU score calculation
        return corpus_bleu(refs, hyps, smoothing_function=smoothie) # Calculate BLEU-4 score
    return 0.0 # Return 0 if NLTK is not available or no hypotheses

best_bleu = 0.0 # Initialize best BLEU score
patience = 5 # Number of epochs to wait for improvement before early stopping
patience_counter = 0 # Counter for patience

# Ensure the save directory for checkpoints exists
os.makedirs(CFG['save_dir'], exist_ok=True)
ckpt_path = Path(CFG['save_dir']) / f"{CFG['exp_name']}.pt"

overall_start_time = time.time() # Start overall training timer

for epoch in range(1, CFG['epochs']+1):
    epoch_start_time = time.time() # Start epoch timer
    tr = train_one_epoch(epoch) # Train for one epoch
    bl = evaluate_bleu(sample_limit=1000) # Evaluate BLEU-4 score
    epoch_duration = time.time() - epoch_start_time # Calculate epoch duration
    print(f"Epoch {epoch}: loss={tr:.4f} BLEU-4={bl:.4f} (duration: {epoch_duration:.2f}s)")

    if bl > best_bleu:
        best_bleu = bl # Update best BLEU score
        patience_counter = 0 # Reset patience counter
        torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict(), 'cfg': CFG}, ckpt_path) # Save best model
        print('Saved best model to ->', ckpt_path)
    else:
        patience_counter += 1 # Increment patience counter
        print(f"BLEU-4 did not improve. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"Early stopping triggered after {patience} epochs without improvement.")
            break # Trigger early stopping

total_training_duration = time.time() - overall_start_time # Calculate total training duration
print(f"Total training duration: {total_training_duration:.2f}s")

NameError: name 'CFG' is not defined

## 7) Inference (Greedy / Beam) & Optional Attention Visualization

In [ ]:
# Reload best (optional)
# Load the best model checkpoint if it exists
if ckpt_path.exists():
    st = torch.load(ckpt_path, map_location=device) # Load checkpoint state dictionary
    encoder.load_state_dict(st['encoder']) # Load encoder state
    decoder.load_state_dict(st['decoder']) # Load decoder state
    print('Loaded checkpoint:', ckpt_path)


def show_attention_on_image(img_pil, alphas, grid_hw):
    """
    Visualizes the attention map from the decoder on an input image.
    It overlays the attention weights on the image to show which parts the model focused on.
    Args:
        img_pil (PIL.Image.Image): The original PIL Image object.
        alphas (List[torch.Tensor]): A list of attention weight tensors, typically from the last decoding step.
        grid_hw (Tuple[int, int]): The height and width of the spatial feature grid (e.g., (7, 7)).
    """
    # alphas: list of (T=H*W) tensors for each step; we take last step for demo
    H,W = grid_hw # Grid dimensions for attention map
    if not alphas:
        plt.figure(figsize=(4,4)); plt.imshow(img_pil); plt.axis('off'); return # Display image without attention if alphas are empty
    att = alphas[-1][0].detach().cpu().numpy().reshape(H,W) # Get the last attention map and reshape
    att = (att - att.min())/(att.max()-att.min()+1e-8) # Normalize attention map
    plt.figure(figsize=(4,4))
    plt.imshow(img_pil) # Display the original image
    plt.imshow(att, cmap='jet', alpha=0.35) # Overlay attention map with transparency
    plt.axis('off') # Hide axes


def generate_caption(image_path: str, mode=None, beam_size=None, show=True):
    """
    Generates a caption for a given image using either greedy search or beam search decoding.
    Args:
        image_path (str): The file path to the input image.
        mode (str, optional): Decoding strategy, 'greedy' or 'beam'. Defaults to CFG['decode'] if None.
        beam_size (int, optional): Beam width for beam search. Defaults to CFG['beam_size'] if None.
        show (bool): If True, displays the image and the generated caption with optional attention visualization.
    Returns:
        List[str]: A list of string tokens representing the generated caption.
    """
    mode = mode or CFG['decode'] # Use configured decode mode if not specified
    beam_size = beam_size or CFG['beam_size'] # Use configured beam size if not specified
    # Define image transformations for inference
    tfm = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.CenterCrop((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    img = Image.open(image_path).convert('RGB') # Load image and convert to RGB
    with torch.no_grad(): # Disable gradient calculations
        x = tfm(img).unsqueeze(0).to(device) # Apply transformations and add batch dimension
        bos = vocab.stoi[SPECIAL_TOKENS['bos']] # Get BOS token ID
        eos = vocab.stoi[SPECIAL_TOKENS['eos']] # Get EOS token ID
        feats, grid_hw = encoder(x) # Encode image with spatial encoder
        if mode=='beam':
            ids = decoder.beam_search(feats, bos, eos, beam=beam_size, max_len=CFG['max_len']) # Beam search decoding
            toks = vocab.denumericalize(ids) # Convert IDs to tokens
            alphas=None # No attention visualization for beam search in this implementation
        else:
            # Default to greedy if mode is not 'beam' or explicitly 'greedy'
            ids, alphas = decoder.greedy_decode(feats, bos, eos, max_len=CFG['max_len']) # Greedy decoding with attention
            toks = vocab.denumericalize(ids[0]) # Convert IDs to tokens
    if show:
        if alphas is not None:
            show_attention_on_image(img, alphas, grid_hw) # Show image with attention overlay
        else:
            plt.figure(figsize=(4,4)); plt.imshow(img); plt.axis('off') # Show image without attention
        plt.title('Predicted: ' + ' '.join(toks)); plt.show() # Display predicted caption as title
    return toks

# --- New logic to pick random images and generate captions ---

# 1. Define a list of image paths from val_ds
all_val_image_paths = [pair[0] for pair in val_ds.pairs]

# 2. Select 3-5 random image paths
num_random_images = random.randint(3, 5) # Select between 3 and 5 random images
random_image_paths = random.sample(all_val_image_paths, num_random_images)

print(f"Generating captions for {len(random_image_paths)} random validation images...")

# 3. Loop through the selected random image paths and call generate_caption
for i, img_path in enumerate(random_image_paths):
    print(f"\n--- Generating caption for image {i+1}/{len(random_image_paths)}: {img_path.name} ---")
    generate_caption(str(img_path), mode='beam', beam_size=5, show=True) # Generate and display caption using beam search